### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import ast
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

sys.path.append(os.path.abspath('..'))

from quinex_application.quinex_processing.mongos_store import MongoStore
from quinex_application.quinex_processing.quinex_client import QuinexClient
from quinex_application.quinex_processing.quinex_postprocessing import QuinexPostprocessing
from quinex_application.postprocessing.spatial_normalization import SpatialNormalization
from quinex_application.quinex_processing.openalex_client import OpenAlexClient
from quinex_application.visualisation.metadata_visualisation import VisualisationMetadata

In [ ]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2010")
qpost = QuinexPostprocessing()
spatial = SpatialNormalization()
oalex = OpenAlexClient()
metadata = VisualisationMetadata()

### **Load Analysis from MongoDB into Metadata DF and Data DF**

In [ ]:
analysis = []
for year in range(2025, 2026):
    analysis.append(f"papers_{year}")
    analysis.append(f"papers_{year}_closed")
    analysis.append(f"papers_{year}_abstract_fallback")

analysis += ["papers_wos", "papers_wos_closed", "papers_wos_abstract_fallback"]

meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_path = "../data/quinex/df_quinex_new.parquet"
df_metadata_path = "../data/quinex/df_quinex_metadata_new.parquet"

for ana in analysis:
    a = mongodb.get_collection(ana)
    
    if "_abstract_fallback" not in ana:
        df_quinex_meta = qpost.create_metadata_df(a, meta_df)
        qpost.save_df(df_quinex_meta, df_metadata_path)

    df_quinex = qpost.create_information_df(a, meta_df)
    qpost.save_df(df_quinex, df_path)

### **Adapt Column Types**

In [ ]:
df_quinex_new = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
print('------------------------------')
print(df_quinex_new['unit_normalized'].apply(type).value_counts())
print('------------------------------')
print(df_quinex_new['quantity_normalized'].apply(type).value_counts())

In [ ]:
# quantity_normalized: str → dict
df_quinex['quantity_normalized'] = df_quinex['quantity_normalized'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)


df_quinex['unit_normalized'] = df_quinex['unit_normalized'].apply(
    lambda x: np.array(json.loads(x)) if isinstance(x, str) else np.array(x) if isinstance(x, list) else x
)

default_unit = np.array([{'exponent': None, 'text': None, 'unit_uri': 'http://qudt.org/vocab/unit/UNITLESS', 'year': None}])
df_quinex['unit_normalized'] = df_quinex['unit_normalized'].apply(
    lambda x: default_unit if x is None else x
)

In [ ]:
df_save = df_quinex.copy()
df_save['quantity_normalized'] = df_save['quantity_normalized'].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else x
)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Rerun Spatial Normalization**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df = spatial.get_geo_information_from_spatial_scope(
    df_quinex, 
    nbr_runs_per_model=1, 
    models = [
        "7 - Qwen3-Coder-30B-A3B-Instruct - A code model from August 2025", 
        "0 - Ministral-3-14B-Instruct-2512 - The latest Ministral from Dec.2.2025",
        ]
    )

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
df["spatial_scope_refined"] = df["spatial_scope_refined"].apply(
    lambda x: json.dumps(x) if isinstance(x, dict) else None
).astype("string")

df.to_parquet(df_path, index=False)

### **Rerun Quantity Parsing**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
CHUNK_SIZE = 200_000
CHECKPOINT_DIR = "../data/quinex/checkpoints_new2"
PARSE_TIMEOUT = 5  # seconds per row
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

total = len(df_quinex)
chunks_done = sorted([
    int(f.replace("chunk_", "").replace(".parquet", ""))
    for f in os.listdir(CHECKPOINT_DIR)
    if f.startswith("chunk_") and f.endswith(".parquet")
])

start_row = max(chunks_done) + CHUNK_SIZE if chunks_done else 0
print(f"Bereits abgeschlossene Chunks: {chunks_done}")
print(f"Starte bei Zeile {start_row} von {total}")

In [ ]:
for chunk_start in tqdm(range(start_row, total, CHUNK_SIZE), desc="Chunks"):
    chunk_end = min(chunk_start + CHUNK_SIZE, total)
    chunk = df_quinex.iloc[chunk_start:chunk_end].copy()

    # Process-basierter Timeout: hängende Parses werden nach PARSE_TIMEOUT Sekunden terminiert
    results = qpost.parse_batch_with_timeout(
        chunk["quantity_text"].tolist(),
        timeout=PARSE_TIMEOUT,
        desc=f"Rows {chunk_start}–{chunk_end}"
    )

    chunk["value_normalized_adapted"] = [r[0] for r in results]
    chunk["unit_normalized_adapted"] = [r[1] for r in results]
    chunk["quantity_normalized_adapted"] = [r[2] for r in results]

    # Serialisiere unit_normalized_adapted und quantity_normalized_adapted zu JSON-String (Dicts mit gemischten Typen sind nicht parquet-kompatibel)
    chunk["unit_normalized_adapted"] = chunk["unit_normalized_adapted"].apply(
        lambda x: json.dumps(x) if x is not None else None
    )
    chunk["quantity_normalized_adapted"] = chunk["quantity_normalized_adapted"].apply(
        lambda x: json.dumps(x) if x is not None else None
    )
    chunk.to_parquet(f"{CHECKPOINT_DIR}/chunk_{chunk_start}.parquet", index=True)

In [ ]:
# Alle Chunks zusammenführen
print("Füge alle Chunks zusammen...")
all_chunk_files = sorted(
    [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("chunk_") and f.endswith(".parquet")],
    key=lambda f: int(f.replace("chunk_", "").replace(".parquet", ""))
)
df_parsed = pd.concat([pd.read_parquet(f"{CHECKPOINT_DIR}/{f}") for f in all_chunk_files])

# Deserialisiere unit_normalized_adapted und quantity_normalized_adapted zurück zu Listen von Dicts
df_parsed["unit_normalized_adapted"] = df_parsed["unit_normalized_adapted"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else None
)
df_parsed["quantity_normalized_adapted"] = df_parsed["quantity_normalized_adapted"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else None
)

df_quinex[["value_normalized_adapted", "unit_normalized_adapted", "quantity_normalized_adapted"]] = df_parsed[["value_normalized_adapted", "unit_normalized_adapted", "quantity_normalized_adapted"]]
print(f"Fertig! {len(df_quinex)} Zeilen verarbeitet.")

In [ ]:
df_quinex_save = df_quinex.copy()
df_quinex_save["unit_normalized_adapted"] = df_quinex_save["unit_normalized_adapted"].apply(
    lambda x: json.dumps(x) if x is not None else None
)
df_quinex_save["quantity_normalized_adapted"] = df_quinex_save["quantity_normalized_adapted"].apply(
    lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
)
df_quinex_save.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Rerun Unit Dimension**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
df_quinex = qpost.apply_unit_dimension_parsing(df_quinex)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex.parquet", index=False)

### **Add SJR to Metadata**

In [ ]:
metadata_df = pd.read_parquet("../data/quinex/df_quinex_metadata.parquet")
df_paper = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_paper = df_paper[df_paper["Quinex"] == True]

In [ ]:
 metadata_df = metadata.add_sjr_to_metadata(
    metadata_df,
    sjr_folder="../data/paper/impactfactor",
    source_col="source",
    year_col="publication_year",
    #source_rename_dict=source_rename_dict
)

metadata_df = metadata_df.merge(
    df_paper[['DOI', 'Source title']],
    on='DOI',
    how='left'
)

mask = metadata_df['SJR'].isna()
metadata_fallback = metadata_df.loc[mask].copy()
metadata_fallback = metadata.add_sjr_to_metadata(
    metadata_fallback,
    sjr_folder="../data/paper/impactfactor",
    source_col="Source title",
    year_col="publication_year",
)
metadata_df.loc[mask, 'SJR'] = metadata_fallback['SJR'].values

In [ ]:
metadata_df.to_parquet("../data/quinex/df_quinex_metadata.parquet")

### **Adapt Column Types**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
print("\nPython-Typen in quantity_normalized:")
print(df_quinex["quantity_normalized"].apply(type).value_counts(dropna=False))

print("\nPython-Typen in quantity_normalized_adapted:")
print(df_quinex["quantity_normalized_adapted"].apply(type).value_counts(dropna=False))

In [ ]:
df_quinex["spatial_scope_refined"] = df_quinex["spatial_scope_refined"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else None
)

In [ ]:
def convert_to_array(x):
    if isinstance(x, str):
        try:
            parsed = json.loads(x)  # falls echtes JSON
        except:
            parsed = ast.literal_eval(x)  # fallback für Python-Strings
        return np.array(parsed)
    
    elif x is None:
        return np.array([{
            'exponent': None,
            'text': None,
            'unit_uri': 'http://qudt.org/vocab/unit/UNITLESS',
            'year': None
        }])
    
    return x  # falls schon korrekt

df_quinex["unit_normalized_adapted"] = df_quinex["unit_normalized_adapted"].apply(convert_to_array)

### **Rename Columns**

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

In [ ]:
# Spalten entfernen
df_quinex = df_quinex.drop(columns=[
    'spatial_scope',
    'value_normalized',
    'unit_normalized',
    'quantity_normalized'
])

# Spalten umbenennen
df_quinex = df_quinex.rename(columns={
    'spatial_scope_refined': 'spatial_scope',
    'value_normalized_adapted': 'value_normalized',
    'unit_normalized_adapted': 'unit_normalized',
    'quantity_normalized_adapted': 'quantity_normalized',
    'unit_dimension_adapted': 'unit_dimension'
})

In [ ]:
# 3) gewünschte Reihenfolge definieren
new_order = [
    "DOI",
    "source_url",
    #"Abstract",
    "publication_year",
    "author_countries",
    "entity",
    "property",
    "quantity_text",
    "quote",
    "value_normalized",
    "unit_normalized",
    "unit_dimension",
    "quantity_normalized",
    "temporal_scope",
    "spatial_scope",
    "references",
    "classification"
]

# 4) nur vorhandene Spalten berücksichtigen (robust gegen Missing Columns)
df_quinex = df_quinex[[col for col in new_order if col in df_quinex.columns]]

# Spalten umbenennen
df_quinex = df_quinex.rename(columns={
    'publication_year': 'Publication Year',
    'author_countries': 'Author Origin',
    'entity': 'Entity',
    'property': 'Property',
    'quantity_text': 'Quantity',
    'quote': 'Quote',
    'value_normalized': 'Value Normalized',
    'spatial_scope': 'Spatial Scope',
    'temporal_scope': 'Temporal Scope',
    'unit_normalized': 'Unit Normalized',
    'references': 'References',
    'classification': 'Classification',
})

In [ ]:
df_quinex.to_parquet("../streamlit/duckdb_data/df_quinex_adapted.parquet", index=False)

### **Add Metadata to DF**

In [ ]:
df = pd.read_parquet("../data/paper/paper_dataset.parquet")
doi_liste = df["DOI"].tolist()

all_responses = []
for start in tqdm(range(0, len(doi_liste), 200), desc="DOI-Batches"):
    batch = doi_liste[start:start + 100]
    a = oalex.fetch_by_doi(batch)

    batch_response = oalex.extract_funding_and_institutions(
        a,
        output_path=f"../data/visualisation_files/oalex_data.json"
    )
    all_responses.extend(batch_response)

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")
df_metadata = pd.read_parquet("../data/quinex/df_quinex_metadata.parquet")
df_paper = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_oalex = pd.read_csv("../data/visualisation_files/oalex_data.csv")  

In [ ]:
len(df_quinex)

In [ ]:
# author_countries aus OpenAlex ersetzen und publisher aus paper ergänzen
def normalize_doi(value):
    if pd.isna(value):
        return None
    doi = str(value).strip().lower()
    doi = doi.replace("https://doi.org/", "").replace("http://doi.org/", "").replace("doi.org/", "")
    if doi.startswith("doi:"):
        doi = doi[4:]
    return doi

df_quinex = df_quinex.copy()
df_quinex["_doi_key"] = df_quinex["DOI"].apply(normalize_doi)

if "author_countries" in df_quinex.columns:
    df_quinex = df_quinex.drop(columns=["author_countries"])

if "author_origins" not in df_oalex.columns:
    raise KeyError("Spalte 'author_origins' fehlt in df_oalex.")
if "Publisher" not in df_paper.columns:
    raise KeyError("Spalte 'Publisher' fehlt in df_paper.")

oalex_lookup = (
    df_oalex[["DOI", "author_origins"]]
    .copy()
    .assign(_doi_key=lambda d: d["DOI"].apply(normalize_doi))
    .drop_duplicates(subset=["_doi_key"], keep="first")
    [["_doi_key", "author_origins"]]
    .rename(columns={"author_origins": "author_countries"})
)

paper_lookup = (
    df_paper[["DOI", "Publisher"]]
    .copy()
    .assign(_doi_key=lambda d: d["DOI"].apply(normalize_doi))
    .drop_duplicates(subset=["_doi_key"], keep="first")
    [["_doi_key", "Publisher"]]
    .rename(columns={"Publisher": "publisher"})
)

df_quinex = df_quinex.merge(oalex_lookup, on="_doi_key", how="left")
df_quinex = df_quinex.merge(paper_lookup, on="_doi_key", how="left")
df_quinex = df_quinex.drop(columns=["_doi_key"])

preferred_order = [
    "DOI",
    "source_url",
    "publication_year",
    "author_countries",
    "publisher",
    "entity",
    "property",
    "quantity_text",
    "quote",
    "value_normalized",
    "unit_normalized",
    "unit_dimension",
    "quantity_normalized",
    "temporal_scope",
    "spatial_scope",
    "references",
    "classification",
]
df_quinex = df_quinex[[col for col in preferred_order if col in df_quinex.columns]]

print(df_quinex[["author_countries", "publisher"]].notna().mean().rename("fill_rate"))
df_quinex.head(2)

In [ ]:
df_quinex.to_parquet("../data/quinex/df_quinex_new.parquet", index=False)

In [ ]:
df_quinex = pd.read_parquet("../data/quinex/df_quinex_new.parquet")

In [ ]:
df_quinex["classification"].value_counts()

In [ ]:
metadata_cols = [
    "DOI",
    "publication_year",
    "author_countries",
    "publisher",
    "entity",
    "property",
    "quote",
    "temporal_scope",
    "spatial_scope",
    "references",
    "classification",
]

missing_cols = [c for c in metadata_cols if c not in df_quinex.columns]
if missing_cols:
    print("Fehlende Spalten:", missing_cols)

cols_present = [c for c in metadata_cols if c in df_quinex.columns]

# Standard: nicht-leer = nicht-NaN
non_empty_mask = df_quinex[cols_present].notna()

# Für diese Textspalten zählen auch leere Strings (inkl. Whitespaces) als fehlend
empty_string_cols = ["entity", "property", "quote"]
for col in empty_string_cols:
    if col in cols_present:
        non_empty_mask[col] = non_empty_mask[col] & df_quinex[col].astype(str).str.strip().ne("")

summary = pd.DataFrame({
    "non_empty_count": non_empty_mask.sum(),
    "total_rows": len(df_quinex),
})
summary["non_empty_pct"] = (summary["non_empty_count"] / summary["total_rows"] * 100).round(2)
summary